# Hidden Markov Model from Scratch

Goal: build a Gaussian HMM for SPY returns and keep the math visible enough that the implementation is easy to audit.

What I want from this notebook:
- a simple volatility-bucket Markov baseline,
- Forward/Backward inference,
- Viterbi decoding,
- Baum-Welch training,
- regime interpretation on train/test data.

References I used while checking the derivation: Rabiner (1989), Nguyen & Nguyen (2015), and McGreevy (2021).


## 0. Setup

Imports are kept near the top so the rest of the notebook reads like an experiment rather than a script.


In [ ]:
%pip install -q numpy pandas scipy plotly yfinance ipywidgets

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm
import warnings
warnings.filterwarnings("ignore")

# Project helpers
from utils.data_utils import prepare_ticker_data, get_observation_sequence, time_split
from utils.viz_utils  import (plot_price_with_regimes, plot_regime_distributions,
                               plot_transition_matrix, plot_alpha_beta, plot_gamma,
                               plot_log_likelihood, plot_viterbi_path, plot_regime_stats)
from utils.metrics    import (regime_statistics, regime_persistence_summary,
                               conditional_sharpe, aic, bic, hmm_n_params)

# From-scratch implementation
from hmm_core import GaussianHMM, forward_pass, backward_pass, viterbi, compute_gamma, compute_xi

print("All imports OK")

## 1. Data: SPY Returns

I use daily SPY data because it is liquid, familiar, and enough for a market-regime demo. The HMM is trained on log returns only; volatility and momentum features are kept for plots and the baseline notebook.

Log return:
$$r_t = \ln(P_t / P_{t-1})$$


In [ ]:
TICKER = "SPY"
df = prepare_ticker_data(TICKER, start="2024-01-01")
df_train, df_test = time_split(df, train_frac=0.80)

# HMM input: keep this one-dimensional on purpose.
# A univariate model is easier to inspect and explain in an interview.
X_train = get_observation_sequence(df_train)
X_test  = get_observation_sequence(df_test)
X_all   = get_observation_sequence(df)

print(f"Total observations : {len(X_all)}")
print(f"Train             : {len(X_train)}  ({df_train.index[0].date()} - {df_train.index[-1].date()})")
print(f"Test              : {len(X_test)}   ({df_test.index[0].date()} - {df_test.index[-1].date()})")
print(f"\nReturn stats: mean={X_all.mean()*252:.2%}  ann.vol={X_all.std()*np.sqrt(252):.2%}")


In [ ]:
# Basic data sanity checks before fitting anything.
assert df_train.index.max() < df_test.index.min(), "Train/test split should respect time order"
assert np.isfinite(X_train).all() and np.isfinite(X_test).all(), "Returns should not contain NaN/inf"

print("Vol bucket counts in train set:")
print(df_train["VolRegime"].value_counts().sort_index())


In [ ]:
# Interactive price + realized volatility chart
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=("SPY Adjusted Close", "Daily Log-Return", "20-Day Realised Vol (ann.)"))

fig.add_trace(go.Scatter(x=df.index, y=df["Close"],
    line=dict(color="rgba(150,180,255,0.9)", width=1.2), name="Close"), row=1, col=1)

fig.add_trace(go.Bar(x=df.index, y=df["Returns"]*100,
    marker_color=np.where(df["Returns"] >= 0, "rgba(99,200,140,0.7)", "rgba(240,80,80,0.7)"),
    name="Return (%)"), row=2, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df["RVol_20d"]*100,
    line=dict(color="rgba(255,195,55,0.9)", width=1), name="RVol 20d (%)"), row=3, col=1)

fig.update_layout(title="SPY: Returns and Volatility",
                  height=700, plot_bgcolor="rgba(15,17,22,1)",
                  paper_bgcolor="rgba(15,17,22,1)",
                  font=dict(color="#e0e0e0"), showlegend=True,
                  xaxis3_rangeslider_visible=False)
fig.update_xaxes(showgrid=True, gridcolor="rgba(80,80,80,0.3)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(80,80,80,0.3)")
fig.show()

## 2. Observable Volatility Baseline

Before fitting hidden states, I bucket 20-day realized volatility into Low / Med / High. This is not the final model; it is just a plain baseline to compare against.

Transition estimate:
$$\hat{a}_{ij} = n_{ij} / \sum_k n_{ik}$$


In [ ]:
from utils.data_utils import empirical_transition_matrix

# These labels are directly observed from volatility buckets.
vol_labels = df_train["VolRegime"].values
A_obs, states = empirical_transition_matrix(vol_labels, ["Low","Med","High"])

print("MLE Transition Matrix (Observable-Factor Markov Chain):")
print(pd.DataFrame(A_obs, index=states, columns=states).round(4))

fig = plot_transition_matrix(A_obs, state_names=states, 
                              title="Volatility Bucket Transition Matrix")
fig.show()

In [ ]:
# Regime statistics for observable MC
ret_arr  = df_train["Returns"].values
lab_arr  = df_train["VolRegime"].values

stats_df = regime_statistics(ret_arr, lab_arr)
print(stats_df.to_string(index=False))

fig = plot_regime_distributions(df_train, "VolRegime", 
                                 title="Observable MC: Return Distributions")
fig.show()

In [ ]:
fig = plot_price_with_regimes(df_train, "VolRegime",
                              title="SPY: Observable Volatility Regimes")
fig.show()

## 3. Forward Algorithm

Forward pass answers: after seeing returns up to date `t`, how much probability mass is in each hidden state?

$$\alpha_t(i) = P(O_1, \ldots, O_t, Q_t=S_i \mid \lambda)$$

I use scaled probabilities in code. Without scaling, a long return sequence quickly underflows to zero.


In [ ]:
# Small toy parameter set before training the real model.
N_demo = 3
pi_demo = np.array([0.5, 0.3, 0.2])
A_demo  = np.array([[0.7, 0.2, 0.1],
                     [0.2, 0.6, 0.2],
                     [0.1, 0.3, 0.6]])
mu_demo    = np.array([-0.002, 0.001, -0.005])
sigma_demo = np.array([0.005,  0.012,  0.025])

alpha_demo, scales_demo = forward_pass(X_train[:50], pi_demo, A_demo, mu_demo, sigma_demo)

print(f"alpha shape  : {alpha_demo.shape}  (T=50, N=3)")
print(f"alpha[0]     : {alpha_demo[0].round(6)}  (sums to {alpha_demo[0].sum():.4f})")
print(f"alpha[-1]    : {alpha_demo[-1].round(6)}")
print(f"Log-likelihood (demo): {np.log(scales_demo).sum():.2f}")

## 4. Backward Algorithm

Backward pass looks from the other direction: given state `i` at date `t`, how likely is the future part of the sequence?

$$\beta_t(i) = P(O_{t+1}, \ldots, O_T \mid Q_t=S_i, \lambda)$$

Combining forward and backward gives the posterior state probabilities, usually called `gamma`.


In [ ]:
beta_demo = backward_pass(X_train[:50], A_demo, mu_demo, sigma_demo, scales_demo)
gamma_demo = compute_gamma(alpha_demo, beta_demo)

print(f"beta shape   : {beta_demo.shape}")
print(f"beta[-1]     : {beta_demo[-1].round(6)}  (initialized to 1 then scaled)")
print(f"gamma shape  : {gamma_demo.shape}")
print(f"gamma[0]     : {gamma_demo[0].round(4)}  (sums to {gamma_demo[0].sum():.4f})")

# Sanity check: posterior state probabilities should sum to 1.
assert np.allclose(gamma_demo.sum(axis=1), 1.0, atol=1e-8), "gamma should sum to 1!"
print("\nGamma rows sum to 1 for all demo dates")

In [ ]:
fig = plot_alpha_beta(alpha_demo, beta_demo, max_t=50,
                      state_names=["S1","S2","S3"])
fig.show()

## 5. Viterbi Algorithm

`gamma` gives a soft state distribution at each date. Viterbi gives one most likely path through the hidden states.

The implementation uses log probabilities so products of many small probabilities do not underflow.


In [ ]:
# Viterbi on demo parameters
path_demo, log_prob_demo = viterbi(X_train[:100], pi_demo, A_demo, mu_demo, sigma_demo)
print(f"Viterbi path (first 20): {path_demo[:20]}")
print(f"Log-probability of best path: {log_prob_demo:.2f}")

## 6. Baum-Welch EM

Baum-Welch is EM for HMMs. The E-step computes expected state and transition occupancy; the M-step updates initial probabilities, transition probabilities, means, and variances.

I use three states as the main run because it maps cleanly to calm / normal / stress. The AIC/BIC check later tests whether this is too simple.


In [ ]:
# Main model: three states is interpretable without becoming hard to label.
hmm = GaussianHMM(n_states=3, n_iter=300, tol=1e-7, random_state=42)
print("Training 3-state Gaussian HMM on SPY daily returns...")
hmm.fit(X_train)
print()
print(hmm.summary())


In [ ]:
# Quick checks after training.
# Numeric state ids are arbitrary, so I sort/interpret states by volatility later.
print("Transition row sums:", np.round(hmm.A_.sum(axis=1), 4))
print("State std devs      :", np.round(hmm.sigma_, 6))


In [ ]:
fig = plot_log_likelihood(hmm.log_likelihoods_,
                          title="Baum-Welch Convergence: 3-State HMM")
fig.show()

## 7. Regime Analysis

Now I attach the learned states back to dates and ask whether they look financially interpretable: return, volatility, Sharpe, and how often each state appears.


In [ ]:
# Two useful views of the same model:
# - states_gamma: most likely state at each date from posterior probabilities
# - states_viterbi: one globally most likely path through time
gamma_train = hmm.predict_proba(X_train)
states_gamma = hmm.predict(X_train)
states_viterbi, log_p_viterbi = hmm.predict_viterbi(X_train)

state_names = ["Low-Vol", "Med-Vol", "High-Vol"]

print("Posterior argmax counts:", dict(zip(*np.unique(states_gamma, return_counts=True))))
print("Viterbi path counts     :", dict(zip(*np.unique(states_viterbi, return_counts=True))))
print("Smallest state share    :", f"{np.bincount(states_gamma).min() / len(states_gamma):.2%}")


In [ ]:
# State occupancy heat map over time
fig = plot_gamma(gamma_train, dates=df_train.index, state_names=state_names)
fig.show()

In [ ]:
# Assign regime labels to df_train
df_train = df_train.copy()
df_train["HMM_State"] = states_gamma

fig = plot_price_with_regimes(df_train, "HMM_State",
                               title="SPY: 3-State HMM States")
fig.show()

In [ ]:
fig = plot_regime_distributions(df_train, "HMM_State",
                                 title="HMM: Regime Return Distributions")
fig.show()

In [ ]:
fig = plot_transition_matrix(hmm.A_, state_names=state_names,
                             title="Learned Transition Matrix A")
fig.show()

In [ ]:
# Soft vs Viterbi comparison
df_vit = df_train.copy()
df_vit["Viterbi_State"] = states_viterbi

fig = plot_viterbi_path(df_vit, states_viterbi, state_names=state_names,
                         title="Viterbi Path: SPY")
fig.show()

In [ ]:
# Regime statistics
stats_hmm = regime_statistics(df_train["Returns"].values, states_gamma)
print(stats_hmm.round(4).to_string(index=False))

In [ ]:
persistence = regime_persistence_summary(states_gamma)
print(persistence.to_string(index=False))

In [ ]:
sharpes = conditional_sharpe(df_train["Returns"].values, states_gamma)
print("Conditional annualized Sharpe per state:")
for s, sh in sharpes.items():
    print(f"  State {s} ({state_names[s]}): {sh:.3f}")

## 8. Number of States

I fit 2-5 states and compare AIC/BIC plus test log-likelihood. AIC/BIC can prefer extra states, but I still check whether the added states are interpretable.

$$\text{AIC} = 2k - 2\mathcal{L}, \quad \text{BIC} = k \ln T - 2\mathcal{L}$$


In [ ]:
results = []
hmm_models = {}

# I stop at 5 states because beyond that the labels become hard to explain.
for N in range(2, 6):
    print(f"  Fitting {N}-state HMM...", end=" ")
    h = GaussianHMM(n_states=N, n_iter=300, tol=1e-7, random_state=42)
    h.fit(X_train)
    ll_train = h.log_likelihood(X_train)
    ll_test  = h.log_likelihood(X_test)
    k        = hmm_n_params(N)
    results.append({
        "N States": N,
        "Train LL":  round(ll_train, 1),
        "Test LL":   round(ll_test, 1),
        "AIC":       round(aic(ll_train, k), 1),
        "BIC":       round(bic(ll_train, k, len(X_train)), 1),
        "# Params":  k,
    })
    hmm_models[N] = h
    print(f"LL={ll_train:.1f}  AIC={aic(ll_train,k):.1f}  BIC={bic(ll_train,k,len(X_train)):.1f}")

print()
print(pd.DataFrame(results).to_string(index=False))

In [ ]:
from utils.viz_utils import plot_aic_bic

model_names = [f"{r['N States']}-state" for r in results]
aics = [r["AIC"] for r in results]
bics = [r["BIC"] for r in results]

fig = plot_aic_bic(model_names, aics, bics, title="HMM Model Selection: AIC vs BIC")
fig.show()

## 9. Out-of-Sample Check

The train/test split is chronological. This is not a trading backtest, but it catches the most obvious overfitting failure mode: regimes that only make sense in-sample.


In [ ]:
df_test = df_test.copy()
df_test["HMM_State"] = hmm.predict(X_test)

fig = plot_price_with_regimes(df_test, "HMM_State",
                               title="SPY Test Set: HMM States")
fig.show()

stats_test = regime_statistics(df_test["Returns"].values, df_test["HMM_State"].values)
print("\nOut-of-sample regime statistics:")
print(stats_test.round(4).to_string(index=False))

## 10. Limitations

- Gaussian emissions are a simplification; financial returns are fat-tailed.
- State numbers can switch across runs, so I interpret states by volatility and return statistics.
- AIC/BIC may reward extra states that are not useful for explanation.
- This is a regime-detection demo, not a trading strategy by itself.


## 11. What This Notebook Keeps

| Component | Implementation |
|---|---|
| Gaussian emission | `hmm_core._emission_matrix` |
| Forward pass | `hmm_core.forward_pass` |
| Backward pass | `hmm_core.backward_pass` |
| State probabilities | `hmm_core.compute_gamma` |
| Transition probabilities | `hmm_core.compute_xi` |
| Viterbi path | `hmm_core.viterbi` |
| Baum-Welch training | `GaussianHMM.fit` |
| Model selection | `utils.metrics` |

Next: `02_baseline_models/02_baseline_models.ipynb` compares this HMM against compact GMM, K-Means, and Isolation Forest baselines.
